In [12]:
# Load network and list all TLS programs
import sumolib
import os

notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
project_root = os.path.dirname(notebook_dir)

net_file = os.path.join(project_root, "sumo", "network", "osm.net.xml.gz")
net = sumolib.net.readNet(net_file)

for tls in net.getTrafficLights():
    programs = tls.getPrograms()
    print(f"\n=== TLS: {tls.getID()} ===")
    for prog_id, prog in programs.items():
       print(f" Program: {prog_id}")
       phases = prog.getPhases()
       for i, phase in enumerate(phases):
         print(f" Phase {i}: state={phase.state} dur={phase.duration}s "
      f"minDur={phase.minDur}s maxDur={phase.maxDur}s")



=== TLS: cluster_29604747_393551251_393552061_394502530_#1more ===

=== TLS: cluster_29696843_393551252_393552059_394512440_#1more ===

=== TLS: cluster_11149179983_29604800_393551254_393552055_#1more ===

=== TLS: cluster_1369085152_1369085197_29604707_393551250_#1more ===

=== TLS: cluster_21631723_393551231_393552095_401526123_#1more ===

=== TLS: cluster_25629241_25629242_393547020_393547032_#4more ===

=== TLS: cluster_2675699373_2675699383_2675699394_2675699404_#3more ===

=== TLS: cluster_29604657_3240419088_3240419089_393551241_#1more ===

=== TLS: cluster_34600564_391169142_391175820_393546103_#5more ===

=== TLS: joinedS_cluster_29605016_769486906_cluster_29605017_769486879 ===

=== TLS: 29604669 ===

=== TLS: cluster_29604992_393547022_402129043_402129047_#1more ===


In [13]:
# For each TLS list connections to understand what each character means
for tls in net.getTrafficLights():
    print(f"\n=== TLS: {tls.getID()} connections ===")
    connections = tls.getConnections()
    for i, (inLane, outLane, linkIdx) in enumerate(connections):
       inEdge = inLane.getEdge().getID()
       outEdge = outLane.getEdge().getID()
       print(f" Link {i}: {inEdge}/{inLane.getIndex()} -> {outEdge}/{outLane.getIndex()}")


=== TLS: cluster_29604747_393551251_393552061_394502530_#1more connections ===
 Link 0: -1263683889#1/0 -> 222151619#1/0
 Link 1: -1263683889#1/0 -> -1263683890#1/0
 Link 2: -1263683889#1/1 -> -285248539#6/1
 Link 3: -1263683889#1/1 -> 1263683889#1/0
 Link 4: -222151619#1/0 -> -1263683890#1/0
 Link 5: -222151619#1/0 -> -285248539#6/0
 Link 6: -222151619#1/1 -> -285248539#6/1
 Link 7: -222151619#1/1 -> 1263683889#1/0
 Link 8: -222151619#1/1 -> 222151619#1/1
 Link 9: 1263683890#1/0 -> -285248539#6/0
 Link 10: 1263683890#1/0 -> 1263683889#1/0
 Link 11: 1263683890#1/1 -> 222151619#1/1
 Link 12: 1263683890#1/1 -> -1263683890#1/0
 Link 13: 285248539#6/0 -> 1263683889#1/0
 Link 14: 285248539#6/0 -> 222151619#1/0
 Link 15: 285248539#6/1 -> 222151619#1/1
 Link 16: 285248539#6/1 -> -1263683890#1/0
 Link 17: 285248539#6/1 -> -285248539#6/1

=== TLS: cluster_29696843_393551252_393552059_394512440_#1more connections ===
 Link 0: -222151618#1/0 -> -1176470383#0/0
 Link 1: -222151618#1/0 -> -2221516

In [17]:
# Check if any phases have conflicting movements (both green)
for tls in net.getTrafficLights():
    programs = tls.getPrograms()
    for prog_id, prog in programs.items():
           for i, phase in enumerate(prog.getPhases()):
               n_green = phase.state.count('G') + phase.state.count('g')
               n_total = len(phase.state)
               if n_green > n_total * 0.7:
                   print(f" WARNING: TLS {tls.getID()} phase {i} has {n_green}/{n_total} green - likely unsafe")

In [21]:
# Verify Overrides loaded correctly
from lxml import etree
import gzip

tls_file = os.path.join(project_root, "sumo", "network", "tls_overrides.add.xml.gz")
with gzip.open(tls_file, "rb") as f:
    tree = etree.parse(f)
root = tree.getroot()
for tl in root.findall("tlLogic"):
    print(f"\nTLS {tl.get('id')} (program: {tl.get('programID')})")
    total_dur = 0
    for phase in tl.findall("phase"):
       dur = int(phase.get("duration"))
       total_dur += dur
       state = phase.get("state")
       n_green = state.count('G') + state.count('g')
       print(f" dur={dur:3d}s greens={n_green:2d}/{len(state)} state={state[:40]}...")
    print(f" Total cycle: {total_dur}s")
 


TLS 29604669 (program: baseline_fixed)
 dur= 30s greens= 1/3 state=Grr...
 dur=  3s greens= 0/3 state=yrr...
 dur=  2s greens= 0/3 state=rrr...
 dur= 20s greens= 2/3 state=rGG...
 dur=  3s greens= 0/3 state=ryy...
 dur=  2s greens= 0/3 state=rrr...
 Total cycle: 60s

TLS cluster_11149179983_29604800_393551254_393552055_#1more (program: baseline_fixed)
 dur= 30s greens= 8/18 state=GGGGrrrGGGGrrrrrrr...
 dur=  3s greens= 0/18 state=yyyyrrryyyyrrrrrrr...
 dur=  2s greens= 0/18 state=rrrrrrrrrrrrrrrrrr...
 dur= 20s greens= 6/18 state=rrrrGGGrrrrGGGrrrr...
 dur=  3s greens= 0/18 state=rrrryyyrrrryyyrrrr...
 dur=  2s greens= 0/18 state=rrrrrrrrrrrrrrrrrr...
 Total cycle: 60s

TLS cluster_1369085152_1369085197_29604707_393551250_#1more (program: baseline_fixed)
 dur= 30s greens= 6/14 state=rrGGGrrGGGrrrr...
 dur=  3s greens= 0/14 state=rryyyrryyyrrrr...
 dur=  2s greens= 0/14 state=rrrrrrrrrrrrrr...
 dur= 20s greens= 4/14 state=GGrrrGGrrrrrrr...
 dur=  3s greens= 0/14 state=yyrrryyrrrrrrr...